# IBD - Lab 06
## Wordcount en Spark

### Configuración

Configuremos Spark en su entorno de Colab. ¡Ejecuta la celda de abajo!

In [1]:
# 1) Clean install
# !pip uninstall -y pyspark findspark -q
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip install -q pyspark
!pip install -q gdown


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
# 2) Set Java path
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# Optional: check Java
!java -version
!echo $JAVA_HOME

openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)
/usr/lib/jvm/java-17-openjdk-amd64


Ahora autenticamos un cliente de Google Drive para descargar el archivo que procesaremos en nuestro trabajo de Spark.

**Asegúrese de seguir las instrucciones interactivas.**

In [4]:
!pip install PyDrive
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [11]:

!gdown --id 1SE6k_0YukzGd5wK-E4i6mG83nydlfvSa -O pg100.txt


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1SE6k_0YukzGd5wK-E4i6mG83nydlfvSa
To: /content/pg100.txt
100% 5.59M/5.59M [00:00<00:00, 136MB/s]


Si ejecutó las celdas anteriores, debería poder ver el archivo *pg100.txt* en la pestaña "Archivos" en el panel izquierdo.

### Tu tarea

Si ejecuta correctamente la etapa de configuración, estará listo para trabajar en el archivo *pg100.txt* que contiene una copia de las obras completas de Shakespeare.

In [12]:
# 3) Start Spark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("firstspark")
    .master("local[*]")
    .config("spark.ui.port", "4050")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", spark.version)
print("SparkContext OK:", sc.appName)

Spark version: 4.0.2
SparkContext OK: firstspark


In [13]:
spark

In [14]:
# YOUR
txt = spark.read.text("pg100.txt")
type(txt)
txt.printSchema()
txt.show(truncate=False)

root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------+
|value                                                                       |
+----------------------------------------------------------------------------+
|The Project Gutenberg EBook of The Complete Works of William Shakespeare, by|
|William Shakespeare                                                         |
|                                                                            |
|This eBook is for the use of anyone anywhere at no cost and with            |
|almost no restrictions whatsoever.  You may copy it, give it away or        |
|re-use it under the terms of the Project Gutenberg License included         |
|with this eBook or online at www.gutenberg.org                              |
|                                                                            |
|** This is a COPYRIGHTED Project Gutenberg eBook, Details Below **          |
|**     P

## Aquí explicaré el método clásico basado en SQL para contar las palabras en el libro correspondiente.



This cell imports Spark SQL functions and converts the text to lowercase.

**Steps**:

Selects the column value (which contains the text lines) and converts all characters to lowercase.

Renames the new column as words_lower.

**Goal**:
Normalize the text so that "Word" and "word" are treated as the same word.

In [15]:
from pyspark.sql import functions as F
txt_lower = txt.select(F.lower(F.col("value")).alias("words_lower"))
txt_lower.printSchema()
txt_lower.show(truncate=False)


root
 |-- words_lower: string (nullable = true)

+----------------------------------------------------------------------------+
|words_lower                                                                 |
+----------------------------------------------------------------------------+
|the project gutenberg ebook of the complete works of william shakespeare, by|
|william shakespeare                                                         |
|                                                                            |
|this ebook is for the use of anyone anywhere at no cost and with            |
|almost no restrictions whatsoever.  you may copy it, give it away or        |
|re-use it under the terms of the project gutenberg license included         |
|with this ebook or online at www.gutenberg.org                              |
|                                                                            |
|** this is a copyrighted project gutenberg ebook, details below **          |
|**

**Description**

This cell splits each line of text into a list of words.

**Steps**:

split(..., " ")
Splits the text using space as separator.

**Example**:

"this is a book"

becomes

["this","is","a","book"]

The result is stored in a column called words_split.

**Goal**:
Convert each text line into an array of words.

In [16]:
textExplodedDf= txt_lower.select(F.split(F.col("words_lower"), " ").alias("words_split")) # lower letters
textExplodedDf.show(truncate=False)

+-----------------------------------------------------------------------------------------+
|words_split                                                                              |
+-----------------------------------------------------------------------------------------+
|[the, project, gutenberg, ebook, of, the, complete, works, of, william, shakespeare,, by]|
|[william, shakespeare]                                                                   |
|[]                                                                                       |
|[this, ebook, is, for, the, use, of, anyone, anywhere, at, no, cost, and, with]          |
|[almost, no, restrictions, whatsoever., , you, may, copy, it,, give, it, away, or]       |
|[re-use, it, under, the, terms, of, the, project, gutenberg, license, included]          |
|[with, this, ebook, or, online, at, www.gutenberg.org]                                   |
|[]                                                                             

**Description**

This cell converts the array of words into individual rows.

**Steps**:

explode()
Takes each word from the array and creates a separate row.

**Example**:

["this","is","a","book"]

becomes

this
is
a
book

Each word becomes one row in the DataFrame.

count()
Counts the total number of words.

**Goal**:
Transform the dataset into a structure suitable for word counting.

In [17]:
textExplodedDf = textExplodedDf.select(F.explode(F.col("words_split")).alias("word")) # Get Individual Words
textExplodedDf.show()
textExplodedDf.count()


+------------+
|        word|
+------------+
|         the|
|     project|
|   gutenberg|
|       ebook|
|          of|
|         the|
|    complete|
|       works|
|          of|
|     william|
|shakespeare,|
|          by|
|     william|
| shakespeare|
|            |
|        this|
|       ebook|
|          is|
|         for|
|         the|
+------------+
only showing top 20 rows


1410671

**Description**

This cell removes empty words.

**Steps**:

ltrim() removes leading spaces.

The condition != "" removes empty strings.

**Example removed values**:

""
" "

**Goal**:
Clean the dataset by removing invalid or empty tokens.

In [ ]:
textExplodedDf = textExplodedDf.where(F.ltrim(F.col("word")) != "")  # Remove Empty Words
textExplodedDf.show()


**Description**

This cell removes punctuation using a regular expression.

**Steps**:

regexp_extract() extracts only alphabetic characters.

**Regex used**:

[a-z]+

**Meaning**:

match one or more lowercase letters

**Example transformations**:

Original	Result
book.	book
hello!	hello
word,	word

**Goal**:
Remove punctuation marks and symbols from words.

In [18]:
textExplodedDf = textExplodedDf.select(F.regexp_extract(F.col("word"), "[a-z]+", 0).alias("word")) #Remove Trailing Punctuation Marks
textExplodedDf.show()

+-----------+
|       word|
+-----------+
|        the|
|    project|
|  gutenberg|
|      ebook|
|         of|
|        the|
|   complete|
|      works|
|         of|
|    william|
|shakespeare|
|         by|
|    william|
|shakespeare|
|           |
|       this|
|      ebook|
|         is|
|        for|
|        the|
+-----------+
only showing top 20 rows


**Description**

This cell performs the Word Count operation.

**Steps**:

groupBy("word")
Groups identical words.

count()
Counts how many times each word appears.

orderBy(desc)
Sorts the results by frequency (most frequent first).

**Example output**:

word	count
the	12000
and	9500
of	8700

**Goal**:
Find the most frequent words in the text.

In [19]:
textWordCounts = textExplodedDf.groupBy("word").count().orderBy(F.col("count").desc()) #Get a Count of Each Word
textWordCounts.show()

+----+------+
|word| count|
+----+------+
|    |508156|
| the| 27826|
| and| 26796|
|   i| 22503|
|  to| 19870|
|  of| 18297|
|   a| 14722|
| you| 13921|
|  my| 12481|
|that| 11540|
|  in| 11146|
|  is|  9805|
| not|  8755|
| for|  8339|
|with|  8059|
|  me|  7778|
|  it|  7742|
|  be|  7143|
|this|  6898|
|your|  6888|
+----+------+
only showing top 20 rows


**Description**

This cell filters the word count results for specific words.

It searches for the frequencies of:

author

title

beer

**Example output**:

word	count
author	12
title	4
beer	2

**Goal**:
Check the occurrence frequency of specific words in the dataset.

In [20]:
textWordCounts.where((F.col("word") == "author") | (F.col("word") == "title") | (F.col("word") == "beer")).show()

+------+-----+
|  word|count|
+------+-----+
| title|   92|
|author|   18|
|  beer|    6|
+------+-----+



# **Task**

Rewrite the previous Word Count program, originally implemented with PySpark DataFrames, using the MapReduce model with PySpark RDDs.

Write a Spark application that calculates the number of words beginning with each letter. Specifically, we want to count the total number of (non-unique) words that begin with a specific letter for each letter. In your implementation, **ignore case sensitivity**; that is, treat all words as lowercase. Additionally, you can ignore any words **that begin** with a non-alphabetic character.

Your implementation must perform the same preprocessing and counting steps, and must produce the same final results.

In [29]:
import re
# Write a MapReduce program to perform a word count on the previously provided text file.

#llegim arxiu de text
txt = spark.read.text("pg100.txt")

#passem a minúscules
from pyspark.sql import functions as F
txt_lower = txt.select(F.lower(F.col("value")).alias("words_lower"))

#separem línies en paraules i les posem en files individuals
df_individual = txt_lower.select(F.explode(F.split(F.col("words_lower"), " ")).alias("word"))

#extraiem 1ra lletra, comprovem q estigui entre la a-z i contem i ordenem
resultat = df_individual.select(F.regexp_extract(F.col("word"), "^([a-z])", 1).alias("lletra")) \
                        .where(F.col("lletra") != "").groupBy("lletra").count().orderBy("lletra")
#mostrem el resultat
resultat.show(30)

+------+------+
|lletra| count|
+------+------+
|     a| 84836|
|     b| 45455|
|     c| 34567|
|     d| 29713|
|     e| 18697|
|     f| 36814|
|     g| 20782|
|     h| 60563|
|     i| 62167|
|     j|  3339|
|     k|  9418|
|     l| 29569|
|     m| 55676|
|     n| 26759|
|     o| 43494|
|     p| 27759|
|     q|  2377|
|     r| 14265|
|     s| 65705|
|     t|123602|
|     u|  9170|
|     v|  5728|
|     w| 59597|
|     x|    14|
|     y| 25855|
|     z|    71|
+------+------+



Una vez que haya obtenido los resultados deseados, **diríjase a moodle y envíe su solución para este laboratorio**.